In [1]:
import paperscraper

In [3]:
covid19 = ['COVID-19', 'SARS-CoV-2']
ai = ['Artificial intelligence', 'Deep learning', 'Machine learning']
mi = ['Medical imaging']
query = [covid19, ai, mi]

In [ ]:
from paperscraper.arxiv import get_and_dump_arxiv_papers
get_and_dump_arxiv_papers(query, output_filepath='covid19_ai_imaging.jsonl')

In [5]:
from paperscraper.pdf import save_pdf
paper_data = {'doi': "10.48550/arXiv.2207.03928"}
save_pdf(paper_data, filepath='gt4sd_paper.pdf')

In [3]:
from paperscraper.get_dumps import arxiv
arxiv(start_date='2020-01-01', end_date='2020-01-03') # scrapes all metadata from 2024 until today.

Fetching 2020-01-03: 100%|██████████| 3/3 [00:32<00:00, 10.74s/it]


In [ ]:
from paperscraper.arxiv import get_and_dump_arxiv_papers
get_and_dump_arxiv_papers(keywords=["machine learning"],output_filepath='local_dump_test2.jsonl', backend='local')

INFO:paperscraper.arxiv.arxiv:Loaded arxiv dump with 843 entries


In [33]:
get_and_dump_arxiv_papers(keywords=["Machine learning", "Deep learning"],output_filepath='local_dump.jsonl', backend='local')

In [ ]:
keywords = []
with open("data/keywords_final.txt", "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:  # make sure it's not an empty line
            keywords.append(line)

print(keywords)
keywords_outer = []
keywords_outer.append(keywords)



In [ ]:
from paperscraper.get_dumps import arxiv

# This will scrape all metadata from 2024-01-01 until 2024-01-03.
# Adjust the end_date as needed.
arxiv(start_date='2021-01-01', end_date='2022-12-31')

In [ ]:
from paperscraper.arxiv import get_and_dump_arxiv_papers

get_and_dump_arxiv_papers(
    keywords=keywords_outer, 
    output_filepath='local_dump_all_2021_to_2022.jsonl', 
    backend='local'
)

In [2]:
import json

papers = []
with open("local_dump_all_2019_to_2022.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        paper = json.loads(line)
        papers.append(paper)

# Now papers is a list of dictionaries, each containing paper metadata.

In [ ]:
len(papers)

In [ ]:
print(papers[0])

In [22]:
import os
import re
import json
import requests
from dotenv import load_dotenv

In [23]:
# 1. Load environment variables from .env file
load_dotenv()
SEMANTIC_SCHOLAR_API_KEY = os.getenv("SEMANTIC_SCHOLAR_API_KEY")


In [24]:
def arxiv_doi_to_arxiv_id(doi):
    """
    Given a DOI like '10.48550/arXiv.2301.00011',
    extract the portion after 'arXiv.' and prepend 'ARXIV:'.
    e.g. -> 'ARXIV:2301.00011'

    Returns None if no match is found.
    """
    match = re.search(r'arXiv\.([0-9]+\.[0-9]+)', doi)
    if match:
        return f"ARXIV:{match.group(1)}"
    return None

In [38]:
def chunkify(lst, chunk_size=400):
    """Yield successive chunk_size-sized lists from lst."""
    for i in range(0, len(lst), chunk_size):
        yield lst[i:i + chunk_size]

In [41]:
def get_arxiv_papers_info(papers):
    """
    1. Parse each paper's 'doi' to find the corresponding ARXIV:<id>
    2. Batch them in chunks of up to 400
    3. For each batch, call the Semantic Scholar /paper/batch endpoint
       requesting fields = referenceCount, citationCount, title
    4. Update each paper dict with 'reference_count' and 'citation_count'.
       Default to 0 if a paper is invalid / not found / returned None or error.
    """

    # Collect (paper_dict, ARXIV_ID) for valid DOIs
    paper_id_pairs = []
    for p in papers:
        doi = p.get('doi', '')
        arxiv_id = arxiv_doi_to_arxiv_id(doi)
        if arxiv_id:
            paper_id_pairs.append((p, arxiv_id))

    if not paper_id_pairs:
        print("No valid arXiv DOIs found.")
        return

    url = "https://api.semanticscholar.org/graph/v1/paper/batch"
    headers = {}
    if SEMANTIC_SCHOLAR_API_KEY:
        headers["x-api-key"] = SEMANTIC_SCHOLAR_API_KEY

    params = {"fields": "referenceCount,citationCount,title"}

    # Process in chunks of size 400
    for chunk in chunkify(paper_id_pairs, chunk_size=400):
        ids_to_fetch = [arxiv_id for (_, arxiv_id) in chunk]
        payload = {"ids": ids_to_fetch}

        # POST the batch request
        response = requests.post(url, params=params, headers=headers, json=payload)
        if response.status_code != 200:
            print(f"Request failed with status {response.status_code}\n{response.text}")
            # Optionally continue to next chunk, or break. We choose continue:
            for (paper_dict, _) in chunk:
                paper_dict['reference_count'] = 0
                paper_dict['citation_count'] = 0
            continue

        results = response.json()  # This should be a list, each item possibly a dict or None
        if len(results) != len(chunk):
            print("Warning: The API returned a different number of results than expected.")
            # We can handle that mismatch in a more robust way if needed.

        for (paper_dict, _), result in zip(chunk, results):
            # If result is None or not a dict, set default values
            if not isinstance(result, dict):
                paper_dict['reference_count'] = 0
                paper_dict['citation_count'] = 0
                continue

            # If there's an 'error' field, the API encountered a problem with this ID
            if "error" in result:
                print(f"API returned an error for one ID: {result['error']}")
                paper_dict['reference_count'] = 0
                paper_dict['citation_count'] = 0
                continue

            # Now safely extract the counts
            reference_count = result.get('referenceCount', 0)
            citation_count = result.get('citationCount', 0)
            paper_dict['reference_count'] = reference_count
            paper_dict['citation_count'] = citation_count

            # (Optional) print or log
            print(f"Updated: {paper_dict.get('title','N/A')}")
            print(f"  reference_count={reference_count}, citation_count={citation_count}")
            print("-" * 60)

In [ ]:
get_arxiv_papers_info(papers)

In [45]:
def save_to_jsonl(papers, output_path):
    """
    Write a list of dictionaries to a JSONL file (one JSON object per line).
    """
    with open(output_path, 'w', encoding='utf-8') as f:
        for paper in papers:
            # Convert dict -> JSON string and write it with a trailing newline
            json_line = json.dumps(paper, ensure_ascii=False)
            f.write(json_line + "\n")

In [54]:

from collections import defaultdict
from datetime import date, timedelta
def create_daily_top_5(papers, year, output_path):
    """
    1) Filter papers by the given year.
    2) Group them by their exact 'YYYY-MM-DD' date.
    3) For each day of that year, pick the top 5 by citation_count.
    4) Write them all (day by day) to a JSONL file.
    """
    # Step A: Group papers by "YYYY-MM-DD", for the target year only
    day_to_papers = defaultdict(list)
    
    for paper in papers:
        date_str = paper.get("date", "")  # e.g. '2023-01-15'
        if not date_str:
            continue
        
        # Parse date string into integers (YYYY-MM-DD)
        try:
            y, m, d = date_str.split("-")
            y, m, d = int(y), int(m), int(d)
        except ValueError:
            # If parsing fails, skip
            continue
        
        # Only consider if it matches the requested year
        if y == year:
            day_str = f"{y:04d}-{m:02d}-{d:02d}"
            day_to_papers[day_str].append(paper)
    
    # Step B: Iterate over each day of the given year (Jan 1 - Dec 31)
    start_date = date(year, 1, 1)
    end_date = date(year, 12, 31)
    n_days = (end_date - start_date).days + 1
    
    results_for_year = []
    for i in range(n_days):
        current_day = start_date + timedelta(days=i)
        current_day_str = current_day.strftime("%Y-%m-%d")
        
        # Get all papers for this day
        daily_papers = day_to_papers.get(current_day_str, [])
        
        # Sort descending by citation_count
        daily_papers.sort(key=lambda p: p.get("citation_count", 0), reverse=True)
        
        # Take top 5
        top_5 = daily_papers[:5]
        
        # Accumulate in our results
        results_for_year.extend(top_5)
    
    # Step C: Write results to a JSONL file
    save_to_jsonl(results_for_year, output_path)
    print(f"{len(results_for_year)} total papers written for year {year} -> {output_path}")

In [ ]:
# 1) Write the full updated list to a JSONL file
all_papers_output = "all_papers.jsonl"
save_to_jsonl(papers, all_papers_output)
print(f"Saved all papers to {all_papers_output}")

In [ ]:
# 2) Create daily top 5 for 2023
daily_2023_output = "papers_2023_top5_by_day.jsonl"
create_daily_top_5(papers, 2023, daily_2023_output)
    
# 3) Create daily top 5 for 2024
daily_2024_output = "papers_2024_top5_by_day.jsonl"
create_daily_top_5(papers, 2024, daily_2024_output)

In [ ]:
!pip install PyPDF2
import PyPDF2

In [66]:
from paperscraper.pdf import save_pdf

def download_pdfs(jsonl_path, output_dir):
    """
    Reads a JSONL file containing paper dicts (including 'doi'),
    downloads each PDF into 'output_dir', naming each PDF by index or by DOI.
    """
    os.makedirs(output_dir, exist_ok=True)

    with open(jsonl_path, 'r', encoding='utf-8') as f:
        for i, line in enumerate(f):
            paper_data = json.loads(line.strip())
            doi = paper_data.get("doi")
            if not doi:
                continue  # Skip if no DOI
            # Construct a unique filename for the PDF
            # Option 1: Use index i
            pdf_filename = f"paper_{i}.pdf"
            # Option 2 (alternative): use sanitized DOI as filename
            # pdf_filename = doi.replace("/", "_") + ".pdf"

            filepath = os.path.join(output_dir, pdf_filename)

            # 'paper_data' must have at least {'doi': "..."}
            # Download PDF to 'filepath'
            try:
                save_pdf({"doi": doi}, filepath=filepath)
                print(f"Downloaded: {doi} -> {filepath}")
            except Exception as e:
                print(f"Error downloading PDF for DOI {doi}: {e}")

In [67]:
def extract_text_from_pdf(pdf_path):
    """
    Extracts text from a local PDF file using PyPDF2.
    Returns a single string containing all pages' text.
    """
    text_content = []
    try:
        with open(pdf_path, 'rb') as f:
            reader = PyPDF2.PdfReader(f)
            for page in reader.pages:
                page_text = page.extract_text() or ""
                text_content.append(page_text)
        full_text = "\n".join(text_content)
        return full_text
    except Exception as e:
        print(f"Error reading PDF '{pdf_path}': {e}")
        return ""

In [68]:
def build_papers_dataframe(pdf_folder, jsonl_path):
    """
    1) Loops over the same JSONL file to match index -> PDF filename.
    2) Extracts text from each PDF in 'pdf_folder'.
    3) Creates a DataFrame with columns ['doi', 'full_text'].
    """
    records = []

    # We'll assume each line in the JSONL maps to a PDF "paper_<i>.pdf"
    # using the same index as download_pdfs().
    with open(jsonl_path, 'r', encoding='utf-8') as f:
        for i, line in enumerate(f):
            paper_data = json.loads(line.strip())
            doi = paper_data.get("doi", "")
            pdf_filename = f"paper_{i}.pdf"
            pdf_path = os.path.join(pdf_folder, pdf_filename)

            if os.path.isfile(pdf_path):
                full_text = extract_text_from_pdf(pdf_path)
            else:
                # PDF might be missing if download failed
                full_text = ""

            records.append({
                "doi": doi,
                "full_text": full_text
            })

    df = pd.DataFrame(records, columns=["doi", "full_text"])
    return df

In [ ]:
# JSONL paths
jsonl_2023 = "papers_2023_top5_by_day.jsonl"
jsonl_2024 = "papers_2024_top5_by_day.jsonl"

# Directories to store PDFs
dir_2023 = "2023_papers"
dir_2024 = "2024_papers"

########################################
# 1) DOWNLOAD PDFs FROM EACH JSONL
########################################
download_pdfs(jsonl_2023, dir_2023)
download_pdfs(jsonl_2024, dir_2024)

In [ ]:
import pandas as pd

########################################
# 2) BUILD DATAFRAMES FOR EACH YEAR
########################################
df_2023_full_text = build_papers_dataframe(dir_2023, jsonl_2023)
df_2024_full_text = build_papers_dataframe(dir_2024, jsonl_2024)

# Show or save dataframes
print("=== 2023 DataFrame Sample ===")
print(df_2023_full_text.head())

print("=== 2024 DataFrame Sample ===")
print(df_2024_full_text.head())

In [ ]:
import pandas as pd

def remove_invalid_surrogates(text: str) -> str:
    """
    Attempt to re-encode text as UTF-8, replacing or ignoring surrogates.
    """
    # Option A: "ignore" drops bad chars
    return text.encode("utf-8", errors="ignore").decode("utf-8")

    # Option B: "replace" puts '?' in place of bad chars
    # return text.encode("utf-8", errors="replace").decode("utf-8")


# Example: apply to your entire DataFrame
df_2023_full_text = df_2023_full_text.applymap(
    lambda x: remove_invalid_surrogates(x) if isinstance(x, str) else x
)

# Now you should be able to save to CSV without errors
df_2023_full_text.to_csv("2023_full_text.csv", index=False, encoding="utf-8")


In [ ]:
# Example: apply to your entire DataFrame
df_2024_full_text = df_2024_full_text.applymap(
    lambda x: remove_invalid_surrogates(x) if isinstance(x, str) else x
)

# Now you should be able to save to CSV without errors
df_2024_full_text.to_csv("2024_full_text.csv", index=False, encoding="utf-8")

In [ ]:
# test csv read

df_2023_full_text_TEST = pd.read_csv("2023_full_text.csv")
df_2024_full_text_TEST = pd.read_csv("2024_full_text.csv")

df_2023_full_text_TEST

In [ ]:
df_2024_full_text_TEST

In [ ]:
import json
import datetime
import math
import random
from collections import defaultdict

INPUT_FILE = "data/arxiv_papers/all_papers.jsonl"
OUTPUT_FILE = "filtered_papers.jsonl"

all_entries = []
with open(INPUT_FILE, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        data = json.loads(line)
        all_entries.append(data)

# Group papers by year-month
papers_by_month = defaultdict(list)
for entry in all_entries:
    # Some lines might not have "date" or might have invalid format, so wrap in try/except if needed
    try:
        date_str = entry["date"]
        dt = datetime.datetime.strptime(date_str, "%Y-%m-%d")
        year_month = f"{dt.year}-{dt.month:02d}"
        papers_by_month[year_month].append(entry)
    except KeyError:
        # If the JSON line doesn't have 'date' or is invalid, skip or handle differently
        continue

filtered_entries = []
for ym, papers in papers_by_month.items():
    # Sort descending by citation_count, defaulting to 0 if missing
    papers_sorted = sorted(
        papers,
        key=lambda x: x.get("citation_count", 0),
        reverse=True
    )
    
    # Top 10% (at least 1)
    top_10pct_size = max(1, math.ceil(len(papers_sorted) * 0.1))
    top_10pct = papers_sorted[:top_10pct_size]

    # Randomly pick up to 50
    if len(top_10pct) > 50:
        chosen = random.sample(top_10pct, 50)
    else:
        chosen = top_10pct
    
    for c in chosen:
        new_item = {
            "doi": c.get("doi", ""),
            "year_month": ym,
            "title": c.get("title", ""),
            "abstract": c.get("abstract", "")
        }
        filtered_entries.append(new_item)

# (Optional) Sort final results chronologically
def ym_to_key(ym):
    year, month = ym.split("-")
    return (int(year), int(month))

filtered_entries.sort(key=lambda x: ym_to_key(x["year_month"]))

with open(OUTPUT_FILE, "w", encoding="utf-8") as out_f:
    for item in filtered_entries:
        out_f.write(json.dumps(item, ensure_ascii=False))
        out_f.write("\n")

print(f"Done! Wrote {len(filtered_entries)} records to {OUTPUT_FILE}.")


In [ ]:
import json
from datetime import datetime

INPUT_FILE = "filtered_papers.jsonl"         # Has year_month, doi, title, abstract
OUTPUT_FILE = "timeordered_finetune.jsonl"

entries = []
with open(INPUT_FILE, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        data = json.loads(line)
        entries.append(data)

def parse_year_month(ym_str):
    # "2023-01" -> datetime(2023,1,1)
    year, month = ym_str.split("-")
    return datetime(int(year), int(month), 1)

# Sort all entries by ascending year-month
entries.sort(key=lambda x: parse_year_month(x["year_month"]))

with open(OUTPUT_FILE, "w", encoding="utf-8") as out_f:
    for e in entries:
        ym = e["year_month"]
        abstract = e.get("abstract", "")

        # Create text for language modeling
        # We use a small marker for the time stamp, then the abstract only
        text_block = f"<|year_month={ym}|>\nAbstract: {abstract}\n"

        new_line = {
            "text": text_block,
            "metadata": {
                "year_month": ym,
                "doi": e.get("doi", "")
            }
        }
        out_f.write(json.dumps(new_line, ensure_ascii=False))
        out_f.write("\n")

print(f"Done! Wrote time-ordered data to {OUTPUT_FILE}.")
